In [2]:
import sqlite3
import pandas as pd

# 1. CREATE IN-MEMORY DATABASE
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. SCHEMA DEFINITION & SEED DATA
cursor.executescript('''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    region TEXT
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATE,
    total_amount REAL,
    FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
);

INSERT INTO customers VALUES
(1, 'Alice Smith', 'North America'),
(2, 'Bob Jones', 'Europe'),
(3, 'Charlie Brown', 'North America'),
(4, 'Diana Prince', 'Asia');

INSERT INTO orders VALUES
(101, 1, '2024-01-15', 250.00),
(102, 1, '2024-02-20', 450.00),
(103, 2, '2024-01-18', 120.00),
(104, 3, '2024-01-22', 890.00),
(105, 3, '2024-03-05', 310.00),
(106, 4, '2024-02-12', 1100.00),
(107, 2, '2024-03-15', 200.00);
''')

# 3. ADVANCED SQL QUERY (CTEs, Joins & Window Functions)
sql_query = """
WITH CustomerMetrics AS (
    SELECT
        c.customer_id,
        c.name,
        c.region,
        COUNT(o.order_id) AS total_orders,
        SUM(o.total_amount) AS total_spent,
        AVG(o.total_amount) AS avg_order_value
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.name, c.region
)
SELECT
    name,
    region,
    total_orders,
    ROUND(total_spent, 2) AS total_spent,
    ROUND(avg_order_value, 2) AS avg_order_value,
    DENSE_RANK() OVER (PARTITION BY region ORDER BY total_spent DESC) AS regional_rank
FROM CustomerMetrics
ORDER BY total_spent DESC;
"""

# 4. EXECUTE & DISPLAY RESULT
df_sql_results = pd.read_sql_query(sql_query, conn)
print("--- SQL QUERY RESULTS ---")
print(df_sql_results)

--- SQL QUERY RESULTS ---
            name         region  total_orders  total_spent  avg_order_value  \
0  Charlie Brown  North America             2       1200.0            600.0   
1   Diana Prince           Asia             1       1100.0           1100.0   
2    Alice Smith  North America             2        700.0            350.0   
3      Bob Jones         Europe             2        320.0            160.0   

   regional_rank  
0              1  
1              1  
2              2  
3              1  
